# How2Sign CSLR — Improved Training Pipeline v2

**Fixes applied vs notebookc5efacafaa (1).ipynb:**
1. `MAX_TOKENS` raised from 500 → **2000** (fewer `[UNK]` tokens)
2. **Feature normalization** per sample (zero-mean, unit-std) — fixes raw coords mean ~200
3. **Input projection** Dense(128) + LayerNorm before BiLSTM
4. **Conv1D stride=2** temporal downsampling (424 → 212 steps)
5. All **BatchNorm → LayerNorm** (eliminates val_loss spikes on padded sequences)
6. **BiLSTM_2 halved** to 128 units (was 256 = 1.57M params — too big for 6K samples)
7. **Second MHA block** added after BiLSTM_2
8. **Dropout raised** 0.2 → 0.35, **L2 raised** 1e-4 → 5e-4
9. **Dense(256) bottleneck** before output logits
10. **Optimizer**: lr=5e-4, clipnorm=1.0 (was 1e-4, 5.0)
11. **Callbacks** loosened: EarlyStopping patience=8, min_delta=0.5
12. **Beam search** decoding for evaluation (was greedy)
13. **Fresh start** — no pre-loaded weights

In [ ]:
# ============================================================
# 1. DATASET PATHS
# ============================================================
from pathlib import Path

BASE_DIR  = Path(r'/kaggle/input/datasets/hadeelgamal/artifacts2')

CSV_TRAIN = BASE_DIR / 'how2sign_train_subset.csv'
CSV_VAL   = BASE_DIR / 'how2sign_val_subset.csv'
CSV_TEST  = BASE_DIR / 'how2sign_test_subset.csv'

JSON_DIR_TRAIN = BASE_DIR / 'train_features'
JSON_DIR_VAL   = BASE_DIR / 'val_features'
JSON_DIR_TEST  = BASE_DIR / 'test_features'

for label, p in [('BASE_DIR',   BASE_DIR),
                 ('CSV_TRAIN',  CSV_TRAIN),  ('CSV_VAL',  CSV_VAL),  ('CSV_TEST',  CSV_TEST),
                 ('JSON_TRAIN', JSON_DIR_TRAIN), ('JSON_VAL', JSON_DIR_VAL), ('JSON_TEST', JSON_DIR_TEST)]:
    print(f"{label:12s} exists={p.exists()}  → {p}")

## 2. Sequence Length Analysis
Auto-sets `SEQUENCE_LENGTH` to the 95th percentile of actual clip lengths.

In [ ]:
# ============================================================
# 2. SEQUENCE LENGTH ANALYSIS
# ============================================================
import os, glob
import numpy as np

def analyse_lengths(features_dirs):
    lengths = []
    for d in features_dirs:
        for fp in glob.glob(os.path.join(d, '*.npy')):
            try:
                arr = np.load(fp, mmap_mode='r')
                lengths.append(arr.shape[0])
            except Exception:
                pass
    return np.array(lengths, dtype=np.int32)

FEATURE_DIRS = [BASE_DIR / 'train_features', BASE_DIR / 'val_features', BASE_DIR / 'test_features']
lengths = analyse_lengths(FEATURE_DIRS)

if len(lengths) == 0:
    print('No .npy files found. Defaulting SEQUENCE_LENGTH = 424.')
    SEQUENCE_LENGTH = 424
else:
    p95 = int(np.percentile(lengths, 95))
    SEQUENCE_LENGTH = int(np.ceil(p95 / 8) * 8)
    pct = (lengths <= SEQUENCE_LENGTH).mean() * 100
    print(f'Clips : {len(lengths):,}   P95 = {p95}   SEQUENCE_LENGTH = {SEQUENCE_LENGTH}  ({pct:.1f}% preserved)')

NUM_FEATURES = 232
# The Conv1D stride=2 halves the time axis seen by CTC
CTC_INPUT_LENGTH = SEQUENCE_LENGTH // 2
print(f'CTC_INPUT_LENGTH (after stride-2 Conv) = {CTC_INPUT_LENGTH}')

## 3. Basic Data Generator (for tokenizer adaptation)

In [ ]:
# ============================================================
# 3. BASIC GENERATOR (used only for tokenizer)
# ============================================================
from tensorflow.keras.utils import Sequence
import pandas as pd

class How2SignGenerator(Sequence):
    COL_NAME     = 'SENTENCE_NAME'
    COL_NPY      = 'NPY_PATH'
    COL_SENTENCE = 'SENTENCE'

    def __init__(self, csv_path, batch_size=8,
                 sequence_length=SEQUENCE_LENGTH, num_features=NUM_FEATURES):
        csv_path = Path(csv_path)
        self.df  = pd.read_csv(csv_path, on_bad_lines='skip') if csv_path.exists() else pd.DataFrame()
        if len(self.df) and {self.COL_NAME, self.COL_NPY}.issubset(self.df.columns):
            self.df = self.df[self.df[self.COL_NPY].notna()].reset_index(drop=True)
            print(f'  {len(self.df):,} rows  ←  {csv_path.name}')
        self.batch_size      = batch_size
        self.sequence_length = sequence_length
        self.num_features    = num_features

    def _load_npy(self, npy_path):
        out = np.zeros((self.sequence_length, self.num_features), dtype=np.float32)
        try:
            seq = np.load(str(BASE_DIR / npy_path))
            T   = min(len(seq), self.sequence_length)
            out[:T] = seq[:T, :self.num_features]
        except Exception:
            pass
        return out

    def __len__(self):  return max(1, int(np.ceil(len(self.df) / self.batch_size)))

    def __getitem__(self, idx):
        batch = self.df.iloc[idx*self.batch_size : (idx+1)*self.batch_size]
        X = np.stack([self._load_npy(str(r[self.COL_NPY])) for _, r in batch.iterrows()])
        sentences = batch[self.COL_SENTENCE].fillna('').tolist() if self.COL_SENTENCE in batch.columns else ['']*len(batch)
        return X, sentences

    def on_epoch_end(self):
        self.df = self.df.sample(frac=1).reset_index(drop=True)

train_gen = How2SignGenerator(CSV_TRAIN, batch_size=8)
val_gen   = How2SignGenerator(CSV_VAL,   batch_size=8)
test_gen  = How2SignGenerator(CSV_TEST,  batch_size=8)

## 4. Tokenizer — Vocabulary
**FIX: `MAX_TOKENS` raised from 500 → 2000** to reduce `[UNK]` mappings, which artificially inflate WER.

In [ ]:
# ============================================================
# 4. TOKENIZER
# FIX: MAX_TOKENS = 2000 (was 500) — fewer [UNK] tokens
# ============================================================
from tensorflow.keras.layers import TextVectorization
import pandas as pd

train_df = pd.read_csv(CSV_TRAIN, on_bad_lines='skip')
train_sentences = train_df['SENTENCE'].fillna('').tolist()

MAX_TOKENS  = 2000   # FIX: was 500
BLANK_INDEX = 0

tokenizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='int'
)
print('Adapting tokenizer...')
tokenizer.adapt(train_sentences)

vocab = tokenizer.get_vocabulary()
print(f'Vocabulary size : {len(vocab)}')
print(f'Top 15 words    : {vocab[:15]}')

def encode_sentence(text):
    indices = tokenizer([text])[0].numpy()
    return indices + 1   # reserve 0 for CTC blank

def decode_indices(indices):
    words = []
    for idx in indices:
        if idx == 0: continue
        adj = idx - 1
        if 0 <= adj < len(vocab):
            w = vocab[adj]
            if w not in ('', '[UNK]'):
                words.append(w)
    return ' '.join(words)

sample = train_sentences[0]
enc = encode_sentence(sample)
print(f'\nSmoke test:')
print(f'  Original : {sample}')
print(f'  Decoded  : {decode_indices(enc)}')

## 5. HDF5 Data Packer (run once)

In [ ]:
# ============================================================
# 5. HDF5 DATA PACKER  (run once — skip if file exists)
# ============================================================
import h5py
from tqdm.auto import tqdm

h5_path   = Path('/kaggle/working/train_features.h5')
base_path = BASE_DIR / 'train_features'

if not h5_path.exists() and CSV_TRAIN.exists():
    print('Packing .npy files into HDF5 for fast I/O...')
    df = pd.read_csv(CSV_TRAIN)
    with h5py.File(h5_path, 'w') as hf:
        for _, row in tqdm(df.iterrows(), total=len(df)):
            name = str(row.get('SENTENCE_NAME', ''))
            npy  = base_path / f'{name}.npy'
            if npy.exists():
                hf.create_dataset(name, data=np.load(npy), compression='gzip', compression_opts=1)
    print('HDF5 packing complete.')
else:
    print(f'HDF5 already exists or CSV not found — skipping.')

## 6. CTC Data Generator
**FIX: Feature normalization added** — zero-mean, unit-std per sample per feature.  
**FIX: `input_length` set to `CTC_INPUT_LENGTH`** (= `SEQUENCE_LENGTH // 2`) to account for the Conv1D stride=2 in the model.

In [ ]:
# ============================================================
# 6. CTC DATA GENERATOR
# FIX A: per-sample feature normalisation (zero-mean, unit-std)
# FIX B: input_length = CTC_INPUT_LENGTH (SEQUENCE_LENGTH // 2)
#         because Conv1D stride=2 halves the time axis
# ============================================================
import h5py, os
import numpy as np
import tensorflow as tf
import pandas as pd

class How2SignCTCGenerator(tf.keras.utils.Sequence):
    def __init__(self, csv_path, base_path, batch_size=32,
                 sequence_length=SEQUENCE_LENGTH,
                 ctc_input_length=CTC_INPUT_LENGTH,
                 max_label_len=50, augment=False):
        super().__init__()
        try:
            self.df = pd.read_csv(csv_path, sep='\t', on_bad_lines='skip')
            if 'SENTENCE_NAME' not in self.df.columns:
                self.df = pd.read_csv(csv_path, on_bad_lines='skip')
        except Exception as e:
            print(f'Warning: {e}'); self.df = pd.DataFrame()

        self.base_path        = base_path
        self.batch_size       = batch_size
        self.sequence_length  = sequence_length
        self.ctc_input_length = ctc_input_length   # FIX B
        self.max_label_len    = max_label_len
        self.augment          = augment

        h5 = '/kaggle/working/train_features.h5'
        self.use_h5  = os.path.exists(h5)
        self.h5_file = h5py.File(h5, 'r') if self.use_h5 else None
        src = f'HDF5: {h5}' if self.use_h5 else f'.npy: {base_path}'
        print(f'Generator ready — {len(self.df):,} samples, batch={batch_size}, augment={augment}, src={src}')

    def load_frames(self, name):
        if self.use_h5 and self.h5_file and name in self.h5_file:
            return self.h5_file[name][:]
        p = os.path.join(self.base_path, f'{name}.npy')
        return np.load(p) if os.path.exists(p) else np.zeros((1, 232), dtype=np.float32)

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.df.iloc[idx*self.batch_size : (idx+1)*self.batch_size]
        n = len(batch)

        X             = np.zeros((n, self.sequence_length, 232), dtype=np.float32)
        Y             = np.zeros((n, self.max_label_len),        dtype=np.int32)
        # FIX B: report the post-stride length so CTC knows the real output T
        input_lengths = np.full((n, 1), self.ctc_input_length, dtype=np.int32)
        label_lengths = np.zeros((n, 1), dtype=np.int32)

        for i, (_, row) in enumerate(batch.iterrows()):
            name   = str(row.get('SENTENCE_NAME', ''))
            frames = self.load_frames(name)
            T      = min(len(frames), self.sequence_length)

            if T > 0:
                data = frames[:T].astype(np.float32)

                # ── FIX A: per-sample normalisation ──────────
                mean = data.mean(axis=0, keepdims=True)
                std  = data.std(axis=0,  keepdims=True) + 1e-6
                data = (data - mean) / std

                if self.augment:
                    data = data + np.random.normal(0, 0.02, data.shape).astype(np.float32)
                    data = data * np.random.uniform(0.95, 1.05)

                X[i, :T, :] = data

            sentence = str(row.get('SENTENCE', ''))
            encoded  = encode_sentence(sentence)
            L = min(len(encoded), self.max_label_len, self.ctc_input_length - 1)
            L = max(L, 1)
            Y[i, :L]         = np.clip(encoded[:L], 1, MAX_TOKENS)
            label_lengths[i] = L

        inputs = {
            'input'        : X,
            'labels'       : Y,
            'input_length' : input_lengths,
            'label_length' : label_lengths,
        }
        return inputs, np.zeros((n,), dtype=np.float32)

    def __del__(self):
        if self.h5_file:
            try: self.h5_file.close()
            except: pass


train_gen_ctc = How2SignCTCGenerator(CSV_TRAIN, BASE_DIR / 'train_features',
                                     batch_size=32, augment=True)
val_gen_ctc   = How2SignCTCGenerator(CSV_VAL,   BASE_DIR / 'val_features',
                                     batch_size=32, augment=False)

print('\n── Batch 0 shape check ──')
sx, _ = train_gen_ctc[0]
for k, v in sx.items():
    print(f'  {k:15s}: {v.shape}  dtype={v.dtype}')

il = sx['input_length'].flatten()
ll = sx['label_length'].flatten()
print(f'\n  Bad samples (label_len >= ctc_input_len): {(ll >= il).sum()}')

# Sanity: check normalised range
x0 = sx['input'][0]           # (424, 232)
non_zero = x0[np.any(x0 != 0, axis=1)]
if len(non_zero):
    print(f'\n  Normalised feature stats (first sample, non-padded frames):')
    print(f'    mean = {non_zero.mean():.4f}   std = {non_zero.std():.4f}')
    print(f'    min  = {non_zero.min():.4f}   max = {non_zero.max():.4f}')

## 7. Improved CTC Model Architecture

All fixes applied:
- Input projection Dense(128) + LayerNorm
- Conv1D stride=2 temporal downsampling
- BatchNorm → LayerNorm everywhere
- BiLSTM_2 reduced: 256 → 128 units
- Second MHA block after BiLSTM_2
- Dropout 0.2 → 0.35
- L2 1e-4 → 5e-4
- Dense(256) bottleneck before output

In [ ]:
# ============================================================
# 7. IMPROVED CTC MODEL
# ============================================================
import tensorflow as tf
from tensorflow.keras.layers import (
    Dense, Dropout, LSTM, Bidirectional, Input,
    MultiHeadAttention, LayerNormalization,
    Conv1D, Softmax
)
from tensorflow.keras.models import Model

tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())

vocab_size_ctc = MAX_TOKENS + 1
print(f'CTC output classes (incl. blank): {vocab_size_ctc}')

REG = tf.keras.regularizers.l2(5e-4)   # FIX: was 1e-4


# ── Custom CTC Loss Layer ────────────────────────────────────
class CTCLossLayer(tf.keras.layers.Layer):
    def __init__(self, blank_index=0, **kwargs):
        super().__init__(**kwargs)
        self.blank_index = blank_index

    def call(self, inputs):
        y_pred, labels, input_length, label_length = inputs
        y_pred       = tf.cast(y_pred, tf.float32)
        log_probs    = tf.math.log(y_pred + 1e-8)
        log_probs_tm = tf.transpose(log_probs, [1, 0, 2])   # [T, B, C]

        input_len_1d = tf.cast(tf.reshape(input_length, [-1]), tf.int32)
        label_len_1d = tf.cast(tf.reshape(label_length, [-1]), tf.int32)
        label_len_1d = tf.minimum(label_len_1d, input_len_1d)
        label_len_1d = tf.maximum(label_len_1d, 1)

        loss = tf.nn.ctc_loss(
            labels           = labels,
            logits           = log_probs_tm,
            label_length     = label_len_1d,
            logit_length     = input_len_1d,
            logits_time_major= True,
            blank_index      = self.blank_index
        )
        loss = tf.where(tf.math.is_finite(loss), loss, tf.zeros_like(loss))
        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'blank_index': self.blank_index})
        return cfg


# ── Inputs ────────────────────────────────────────────────────
input_frames = Input(shape=(SEQUENCE_LENGTH, 232), name='input',   dtype='float32')
labels_in    = Input(shape=(None,),               name='labels',   dtype='int32')
input_len_in = Input(shape=(1,),                  name='input_length', dtype='int32')
label_len_in = Input(shape=(1,),                  name='label_length', dtype='int32')

# ── FIX 3: Input projection — compact feature embedding ──────
x = Dense(128, activation='relu', name='input_proj', dtype='float32')(input_frames)
x = LayerNormalization(name='input_ln')(x)

# ── FIX 4: Temporal downsampling — 424 → 212 steps ───────────
x = Conv1D(128, kernel_size=3, strides=2, padding='same',
           activation='relu', name='temporal_conv')(x)
x = LayerNormalization(name='conv_ln')(x)

# ── BiLSTM 1 ─────────────────────────────────────────────────
x1 = Bidirectional(LSTM(256, return_sequences=True, kernel_regularizer=REG),
                   name='bilstm_1')(x)
x1 = LayerNormalization(name='ln_1')(x1)   # FIX 5: was BatchNorm

# ── Attention 1 (with residual) ───────────────────────────────
attn1 = MultiHeadAttention(num_heads=4, key_dim=64, name='mha_1')(x1, x1)
x1    = LayerNormalization(name='ln_mha1')(x1 + attn1)
x1    = Dropout(0.35, name='drop_1')(x1)   # FIX 8: was 0.2

# ── FIX 6: BiLSTM 2 — halved to 128 units ────────────────────
x2 = Bidirectional(LSTM(128, return_sequences=True, kernel_regularizer=REG),
                   name='bilstm_2')(x1)
x2 = LayerNormalization(name='ln_2')(x2)   # FIX 5: was BatchNorm

# ── FIX 7: Attention 2 (new block after BiLSTM 2) ────────────
attn2 = MultiHeadAttention(num_heads=4, key_dim=32, name='mha_2')(x2, x2)
x2    = LayerNormalization(name='ln_mha2')(x2 + attn2)
x2    = Dropout(0.35, name='drop_2')(x2)   # FIX 8: was 0.2

# ── FIX 9: Bottleneck Dense before output ────────────────────
x2 = Dense(256, activation='relu', name='bottleneck', dtype='float32')(x2)
x2 = Dropout(0.2, name='drop_3')(x2)

# ── Output ───────────────────────────────────────────────────
logits = Dense(vocab_size_ctc, name='logits', dtype='float32')(x2)
y_pred = Softmax(name='prediction', dtype='float32')(logits)

# ── CTC Loss ─────────────────────────────────────────────────
ctc_loss_out = CTCLossLayer(blank_index=0, name='ctc_loss')(
    [y_pred, labels_in, input_len_in, label_len_in]
)

# ── Models ───────────────────────────────────────────────────
model_ctc_train = Model(
    inputs  = [input_frames, labels_in, input_len_in, label_len_in],
    outputs = ctc_loss_out,
    name    = 'ctc_train'
)
model_ctc_train.compile(
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=5e-4,   # FIX 10: was 1e-4
        clipnorm=1.0          # FIX 10: was 5.0
    ),
    loss = lambda y_true, y_pred: y_pred
)

model_inference = Model(
    inputs  = input_frames,
    outputs = y_pred,
    name    = 'ctc_inference'
)

model_ctc_train.summary()
model_inference.summary()

## 8. Training
**FIX: Fresh start** — no pre-loaded weights.  
**FIX: Looser callbacks** — EarlyStopping patience=8, min_delta=0.5.

In [ ]:
# ============================================================
# 8. TRAINING
# FIX 11: Fresh start — no pre-loaded weights
# FIX 11: Looser callbacks to let the model converge properly
# ============================================================
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import os

CHECKPOINT_PATH = '/kaggle/working/best_cslr_model_v2.weights.h5'
print(f'Fresh start — training from scratch.')
print(f'Weights will be saved to: {CHECKPOINT_PATH}')

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=8,          # FIX: was 5 — too aggressive for noisy CTC
        restore_best_weights=True,
        min_delta=0.5        # FIX: was 1.0 — too coarse
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,          # FIX: was 3
        min_delta=0.5,       # FIX: was 1.0
        min_lr=1e-6
    ),
    ModelCheckpoint(
        CHECKPOINT_PATH,
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True
    )
]

history = model_ctc_train.fit(
    train_gen_ctc,
    validation_data=val_gen_ctc,
    epochs=50,               # more room to converge
    callbacks=callbacks
)

## 9. Training History Plot

In [ ]:
# ============================================================
# 9. PLOT TRAINING HISTORY
# ============================================================
import matplotlib.pyplot as plt

train_loss = history.history.get('loss', [])
val_loss   = history.history.get('val_loss', [])
lr_history = history.history.get('learning_rate', [])
epochs     = range(1, len(train_loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, train_loss, label='Train loss')
axes[0].plot(epochs, val_loss,   label='Val loss')
axes[0].set_title('CTC Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

if lr_history:
    axes[1].plot(epochs, lr_history)
    axes[1].set_title('Learning Rate')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('LR')
    axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('/kaggle/working/training_history.png', dpi=120)
plt.show()
print(f'Best val_loss : {min(val_loss):.4f} (epoch {val_loss.index(min(val_loss))+1})')

## 10. Evaluation
**FIX: Beam search decoding** (width=10) instead of greedy — gives better WER at the same loss level.

In [ ]:
# ============================================================
# 10. EVALUATION & WER
# FIX 12: beam search decoding (was greedy)
# ============================================================
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'jiwer'])
import jiwer
import tensorflow as tf

test_gen_ctc = How2SignCTCGenerator(
    CSV_TEST, BASE_DIR / 'test_features',
    batch_size=32, augment=False
)

def evaluate_model(model_inf, gen, beam_width=10):
    print(f'Evaluating on {len(gen)*gen.batch_size} samples '
          f'(beam_width={beam_width})...')
    hypotheses, references = [], []

    for i in range(len(gen)):
        batch_x, _ = gen[i]
        logits = model_inf.predict(batch_x['input'], verbose=0)

        # FIX 12: beam search
        input_lens = batch_x['input_length'].flatten()
        decoded, _ = tf.keras.backend.ctc_decode(
            logits, input_length=input_lens,
            greedy=False, beam_width=beam_width
        )
        dec_np = decoded[0].numpy()

        for j in range(len(dec_np)):
            hyp = decode_indices([x for x in dec_np[j] if x != -1])
            ref = decode_indices([x for x in batch_x['labels'][j] if x != 0])
            hypotheses.append(hyp if hyp else 'empty')
            references.append(ref if ref else 'empty')

    wer = jiwer.wer(references, hypotheses)
    print(f'\nWord Error Rate (WER) : {wer:.4f}')
    print(f'Correct word rate    : {1 - wer:.4f}')

    print('\nSample predictions (first 5):')
    for ref, hyp in zip(references[:5], hypotheses[:5]):
        print(f'  REF: {ref}')
        print(f'  HYP: {hyp}')
        print()

    return references, hypotheses

refs, hyps = evaluate_model(model_inference, test_gen_ctc)

## 11. Stabilization Tracker

In [ ]:
# ============================================================
# 11. STABILIZATION TRACKER  (sliding window majority vote)
# ============================================================
from collections import deque
import time

class StabilizationTracker:
    def __init__(self, window_size=15, majority_ratio=0.6, cooldown_s=1.0):
        self.window_size          = window_size
        self.majority_ratio       = majority_ratio
        self.cooldown_s           = cooldown_s
        self.buffer               = deque(maxlen=window_size)
        self.last_commit_time     = 0
        self.last_committed_word  = ''
        self.sentence             = []

    def update(self, predicted_word):
        if not predicted_word:
            self.buffer.append(None)
            return None
        self.buffer.append(predicted_word)
        if len(self.buffer) < self.window_size:
            return None
        counts = {w: 0 for w in self.buffer if w}
        for w in self.buffer:
            if w: counts[w] += 1
        if not counts: return None
        top_word  = max(counts, key=counts.get)
        top_ratio = counts[top_word] / self.window_size
        now = time.time()
        if top_ratio >= self.majority_ratio:
            if top_word != self.last_committed_word and (now - self.last_commit_time) > self.cooldown_s:
                self.sentence.append(top_word)
                self.last_committed_word = top_word
                self.last_commit_time    = now
                self.buffer.clear()
                return top_word
        return None

tracker = StabilizationTracker()
print('StabilizationTracker ready.')

## 12. Real-Time Webcam Inference

In [ ]:
pip install mediapipe==0.10.14

In [ ]:
# ============================================================
# 12. WEBCAM INFERENCE  (232-dim, accounts for stride-2 Conv)
# ============================================================
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
from collections import deque

mp_holistic = mp.solutions.holistic
mp_drawing  = mp.solutions.drawing_utils

_FACE_KP_INDICES = (
    [17,18,19,20,21] + [22,23,24,25,26] +
    [36,37,38,39,40,41] + [42,43,44,45,46,47] +
    [68,69] +
    [27,28,29,30] + [33] +
    [48,49,50,51,52,53,54,55,56,57,58,59] +
    [60,61,62,63,64,65,66,67]
)

_MP_FACE_TO_OP70 = {
    17:70, 18:63, 19:105, 20:66, 21:107,
    22:336, 23:296, 24:334, 25:293, 26:300,
    36:33, 37:160, 38:158, 39:133, 40:153, 41:144,
    42:362, 43:385, 44:387, 45:263, 46:373, 47:380,
    68:468, 69:473,
    27:6, 28:197, 29:195, 30:5, 33:1,
    48:61, 49:185, 50:40, 51:39, 52:37, 53:0,
    54:267, 55:269, 56:270, 57:409, 58:291, 59:375,
    60:78, 61:191, 62:80, 63:81,
    64:311, 65:310, 66:415, 67:308
}

def mediapipe_to_232dim(results):
    pose = np.zeros((25, 2), dtype=np.float32)
    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def sp(op, mp_i):
            if mp_i < len(lm): pose[op] = [lm[mp_i].x, lm[mp_i].y]
        sp(0,0); sp(2,12); sp(3,14); sp(4,16); sp(5,11); sp(6,13); sp(7,15)
        sp(9,24); sp(10,26); sp(11,28); sp(12,23); sp(13,25); sp(14,27)
        sp(15,5); sp(16,2); sp(17,8); sp(18,7); sp(19,31); sp(21,29); sp(22,32); sp(24,30)
        if pose[2].any() and pose[5].any(): pose[1] = (pose[2]+pose[5])/2
        if pose[9].any() and pose[12].any(): pose[8] = (pose[9]+pose[12])/2

    lhand = np.zeros((21,2), dtype=np.float32)
    if results.left_hand_landmarks:
        for k, pt in enumerate(results.left_hand_landmarks.landmark):
            lhand[k] = [pt.x, pt.y]

    rhand = np.zeros((21,2), dtype=np.float32)
    if results.right_hand_landmarks:
        for k, pt in enumerate(results.right_hand_landmarks.landmark):
            rhand[k] = [pt.x, pt.y]

    face = np.zeros((49,2), dtype=np.float32)
    if results.face_landmarks:
        mesh = results.face_landmarks.landmark
        for k, op_idx in enumerate(_FACE_KP_INDICES):
            mp_i = _MP_FACE_TO_OP70.get(op_idx)
            if mp_i is not None and mp_i < len(mesh):
                face[k] = [mesh[mp_i].x, mesh[mp_i].y]

    return np.concatenate([pose.flatten(), lhand.flatten(), rhand.flatten(), face.flatten()])


def run_inference(source=0):
    cap     = cv2.VideoCapture(source)
    tracker = StabilizationTracker(window_size=15, majority_ratio=0.6)
    frame_buffer = deque(maxlen=SEQUENCE_LENGTH)

    with mp_holistic.Holistic(min_detection_confidence=0.5,
                              min_tracking_confidence=0.5) as holistic:
        frame_count = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = holistic.process(img)
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

            feat = mediapipe_to_232dim(res)
            frame_buffer.append(feat)

            if len(frame_buffer) > 10 and frame_count % 5 == 0:
                X = np.expand_dims(np.stack(frame_buffer), axis=0)
                if X.shape[1] < SEQUENCE_LENGTH:
                    pad = np.zeros((1, SEQUENCE_LENGTH - X.shape[1], 232))
                    X = np.concatenate([X, pad], axis=1)

                # Normalise the inference buffer the same way as training
                mean = X.mean(axis=(0,1), keepdims=True)
                std  = X.std(axis=(0,1),  keepdims=True) + 1e-6
                X    = (X - mean) / std

                preds = model_inference.predict(X, verbose=0)
                # CTC_INPUT_LENGTH after stride-2 Conv
                input_lens = np.array([CTC_INPUT_LENGTH])
                decoded, _ = tf.keras.backend.ctc_decode(
                    preds, input_length=input_lens, greedy=True)
                sentence = decode_indices(decoded[0][0].numpy())
                tracker.update(sentence)

            text = ' '.join(tracker.sentence) if tracker.sentence else ''
            cv2.putText(img, text, (10, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
            cv2.imshow('Sign Language Translation', img)

            frame_count += 1
            if cv2.waitKey(10) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()

# run_inference(0)

## 13. Export to ONNX

In [ ]:
pip install tf2onnx

In [ ]:
# ============================================================
# 13. EXPORT TO ONNX
# ============================================================
import tensorflow as tf, tf2onnx, os

model_inference.load_weights(CHECKPOINT_PATH)
print('Loaded best weights for ONNX export.')

onnx_path       = 'cslr_inference_v2.onnx'
saved_model_dir = 'cslr_saved_model_v2'

try:
    model_inference.save(saved_model_dir)
    print('Saved to SavedModel. Converting to ONNX...')
    os.system(
        f'python -m tf2onnx.convert '
        f'--saved-model {saved_model_dir} '
        f'--output {onnx_path} --opset 13'
    )
    print(f'ONNX model saved to {onnx_path}')
except Exception as e1:
    print(f'SavedModel strategy failed ({e1}), trying Keras direct...')
    try:
        sig = [tf.TensorSpec([1, SEQUENCE_LENGTH, 232], tf.float32, name='input')]
        onnx_model, _ = tf2onnx.convert.from_keras(model_inference, sig, opset=13)
        with open(onnx_path, 'wb') as f:
            f.write(onnx_model.SerializeToString())
        print(f'ONNX saved via Keras direct: {onnx_path}')
    except Exception as e2:
        print(f'Both strategies failed: {e2}')

try:
    import onnxruntime
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(onnx_path, 'cslr_v2_int8.onnx', weight_type=QuantType.QInt8)
    print('INT8 quantized model saved to cslr_v2_int8.onnx')
except Exception as e:
    print(f'INT8 quantization skipped: {e}')

## 14. Package & Download

In [ ]:
# ============================================================
# 14. PACKAGE FOR DOWNLOAD
# ============================================================
import os, zipfile, glob

def zip_folder(folder, zf, prefix):
    for fp in sorted(glob.glob(os.path.join(folder, '**', '*'), recursive=True)):
        if os.path.isfile(fp):
            zf.write(fp, prefix + '/' + os.path.relpath(fp, folder))

OUTPUT_ZIP = '/kaggle/working/how2sign_features_export_v2.zip'

FEATURE_DIRS = {
    'train_features': '/kaggle/working/train_features',
    'val_features'  : '/kaggle/working/val_features',
    'test_features' : '/kaggle/working/test_features',
}
CSV_FILES = {
    'how2sign_train_subset.csv': '/kaggle/working/how2sign_train_subset.csv',
    'how2sign_val_subset.csv'  : '/kaggle/working/how2sign_val_subset.csv',
    'how2sign_test_subset.csv' : '/kaggle/working/how2sign_test_subset.csv',
}

print(f'Building archive → {OUTPUT_ZIP}')
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for prefix, folder in FEATURE_DIRS.items():
        if os.path.isdir(folder):
            files = glob.glob(os.path.join(folder, '*.npy'))
            print(f'  {prefix}/  →  {len(files):,} files')
            zip_folder(folder, zf, prefix)
        else:
            print(f'  {prefix}/ not found — skipping')
    for name, path in CSV_FILES.items():
        if os.path.isfile(path):
            zf.write(path, name)
            print(f'  {name}  ({os.path.getsize(path)/1024:.1f} KB)')
        else:
            print(f'  {name} not found — skipping')

size_mb = os.path.getsize(OUTPUT_ZIP) / (1024*1024)
print(f'\nDone!  Archive size: {size_mb:.1f} MB  →  {OUTPUT_ZIP}')
print('Kaggle sidebar → Output → how2sign_features_export_v2.zip → Download')